In [0]:
from pyspark.sql import functions as F 

In [0]:
df = spark.table("skztour.01_bronze.raw_wikipedia_dominate")
display(df)

setlist_id,event_date,last_updated,artist_name,artist_mbid,venue_name,venue_id,venue_city,sets,info,url
dominate_1,August 24,null,Stray Kids,null,KSPO Dome,null,Seoul,null,South Korea,https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_2,August 25,null,Stray Kids,null,KSPO Dome,null,Seoul,null,South Korea,https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_3,August 31,null,Stray Kids,null,KSPO Dome,null,Seoul,null,South Korea,https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_4,September 1 [ A ],null,Stray Kids,null,KSPO Dome,null,Seoul,null,South Korea,https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_5,September 28,null,Stray Kids,null,"30,000 [ 23 ]",null,Singapore,null,National Stadium,https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_6,October 19,null,Stray Kids,null,Marvel Stadium,null,Melbourne,null,Australia,https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_7,October 26,null,Stray Kids,null,Allianz Stadium,null,Sydney,null,Australia,https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_8,November 3 [ B ],null,Stray Kids,null,Kaohsiung National Stadium,null,Kaohsiung,null,Taiwan,https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_9,November 14,null,Stray Kids,null,Tokyo Dome,null,Tokyo,null,Japan,https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_10,November 16,null,Stray Kids,null,Tokyo Dome,null,Tokyo,null,Japan,https://en.wikipedia.org/wiki/Dominate_World_Tour


In [0]:
from pyspark.sql.functions import col, trim, regexp_replace, concat, lit, year, try_to_date

print("Antes da transformação:")
df.select("event_date").distinct().show(10, truncate=False)

# Passo 1: Remover referências [ A ], [ B ], etc
df_with_date = df.withColumn(
    "date_cleaned",
    trim(regexp_replace(col("event_date"), r"\[.*?\]", ""))
)

# Passo 2: Adicionar o ano 2025
df_with_date = df_with_date.withColumn(
    "date_with_year",
    concat(col("date_cleaned"), lit(", 2025"))
)

# Passo 3: Converter para DATE (try_to_date retorna NULL se inválido)
df_with_date = df_with_date.withColumn(
    "event_date_parsed",
    try_to_date(col("date_with_year"), "MMMM d, yyyy")
)

print("\nApós transformação:")
df_with_date.select("event_date", "date_cleaned", "date_with_year", "event_date_parsed").show(5, truncate=False)

# Passo 4: Filtrar linhas com datas válidas e substituir coluna original
df_transformed = df_with_date \
    .filter(col("event_date_parsed").isNotNull()) \
    .drop("event_date", "date_cleaned", "date_with_year") \
    .withColumnRenamed("event_date_parsed", "event_date")

print(f"\nTotal de shows com datas válidas: {df_transformed.count()}")

Antes da transformação:
+-----------------+
|event_date       |
+-----------------+
|August 24        |
|August 25        |
|August 31        |
|September 1 [ A ]|
|September 28     |
|October 19       |
|October 26       |
|November 3 [ B ] |
|November 14      |
|November 16      |
+-----------------+
only showing top 10 rows

Após transformação:
+-----------------+------------+------------------+-----------------+
|event_date       |date_cleaned|date_with_year    |event_date_parsed|
+-----------------+------------+------------------+-----------------+
|August 24        |August 24   |August 24, 2025   |2025-08-24       |
|August 25        |August 25   |August 25, 2025   |2025-08-25       |
|August 31        |August 31   |August 31, 2025   |2025-08-31       |
|September 1 [ A ]|September 1 |September 1, 2025 |2025-09-01       |
|September 28     |September 28|September 28, 2025|2025-09-28       |
+-----------------+------------+------------------+-----------------+
only showing top 5 r

In [0]:
print("Antes da limpeza:")
df_transformed.select("venue_name").distinct().show(10, truncate=False)
df_transformed.select("venue_city").distinct().show(10, truncate=False)

# Aplicar limpeza no df_transformed (que já tem as datas corretas do Cell 5)
df_transformed = df_transformed.withColumn(
    "venue_city",
    trim(regexp_replace(col("venue_city"), r"\[.*?\]", ""))
).withColumn(
    "venue_name", 
    trim(regexp_replace(col("venue_name"), r"\[.*?\]", ""))
)

# Tornar nulo valores numéricos (incluindo números com vírgulas/pontos) ou venue_name igual a "0"
df_transformed = df_transformed.withColumn(
    "venue_city",
    F.when(~F.col("venue_city").rlike("^[0-9,\\.\\s]+$"), F.col("venue_city")).otherwise(F.lit(None))
).withColumn(
    "venue_name",
    F.when(~F.col("venue_name").rlike("^[0-9,\\.\\s]+$") & (F.col("venue_name") != "0"), F.col("venue_name")).otherwise(F.lit(None))
)

print("\nDepois da limpeza:")
df_transformed.select("venue_city").distinct().show(10, truncate=False)
df_transformed.select("venue_name").distinct().show(10, truncate=False)

print(f"\n✅ venue_city limpa! Total de shows: {df_transformed.count()}")

Antes da limpeza:
+--------------------------+
|venue_name                |
+--------------------------+
|KSPO Dome                 |
|NULL                      |
|Marvel Stadium            |
|Allianz Stadium           |
|Kaohsiung National Stadium|
|Tokyo Dome                |
|Philippine Arena          |
|Galaxy Arena              |
|Kyocera Dome              |
|Suphachalasai Stadium     |
+--------------------------+
only showing top 10 rows
+----------+
|venue_city|
+----------+
|Seoul     |
|Singapore |
|Melbourne |
|Sydney    |
|Kaohsiung |
|Tokyo     |
|Bocaue    |
|Macau     |
|Osaka     |
|Bangkok   |
+----------+
only showing top 10 rows

Depois da limpeza:
+----------+
|venue_city|
+----------+
|Seoul     |
|Singapore |
|Melbourne |
|Sydney    |
|Kaohsiung |
|Tokyo     |
|Bocaue    |
|Macau     |
|Osaka     |
|Bangkok   |
+----------+
only showing top 10 rows
+--------------------------+
|venue_name                |
+--------------------------+
|KSPO Dome                 |
|

In [0]:
# 1. Limpar whitespace dos campos de texto 
# 2. Extrair year da data já convertida
# 3. Adicionar coluna source
# 4. Padronizar schema para unificação

df_clean = df_transformed \
    .withColumn("source", F.lit("wikipedia")) \
    .withColumn("artist_name", F.trim(F.col("artist_name"))) \
    .withColumn("venue_name", F.trim(F.col("venue_name"))) \
    .withColumn("venue_city", F.trim(F.col("venue_city"))) \
    .withColumn("event_year", F.year(F.col("event_date"))) \
    .select(
        F.col("setlist_id").alias("show_id"), 
        "source", 
        "event_date", 
        "artist_name", 
        "venue_name", 
        "venue_city", 
        F.col("info").alias("country"),
        F.lit("15,000").cast("string").alias("attendance"), 
        F.col("url").alias("source_url")
    )

print(f"Total Shows: {df_clean.count()}")


Total Shows: 57


In [0]:
# Verificar resultado final com datas transformadas
print("Schema:")
df_clean.printSchema()

Schema:
root
 |-- show_id: string (nullable = true)
 |-- source: string (nullable = false)
 |-- event_date: date (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- venue_name: string (nullable = true)
 |-- venue_city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- attendance: string (nullable = false)
 |-- source_url: string (nullable = true)



In [0]:
display(df_clean)

show_id,source,event_date,artist_name,venue_name,venue_city,country,attendance,source_url
dominate_1,wikipedia,2025-08-24,Stray Kids,KSPO Dome,Seoul,South Korea,"15,000",https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_2,wikipedia,2025-08-25,Stray Kids,KSPO Dome,Seoul,South Korea,"15,000",https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_3,wikipedia,2025-08-31,Stray Kids,KSPO Dome,Seoul,South Korea,"15,000",https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_4,wikipedia,2025-09-01,Stray Kids,KSPO Dome,Seoul,South Korea,"15,000",https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_5,wikipedia,2025-09-28,Stray Kids,null,Singapore,National Stadium,"15,000",https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_6,wikipedia,2025-10-19,Stray Kids,Marvel Stadium,Melbourne,Australia,"15,000",https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_7,wikipedia,2025-10-26,Stray Kids,Allianz Stadium,Sydney,Australia,"15,000",https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_8,wikipedia,2025-11-03,Stray Kids,Kaohsiung National Stadium,Kaohsiung,Taiwan,"15,000",https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_9,wikipedia,2025-11-14,Stray Kids,Tokyo Dome,Tokyo,Japan,"15,000",https://en.wikipedia.org/wiki/Dominate_World_Tour
dominate_10,wikipedia,2025-11-16,Stray Kids,Tokyo Dome,Tokyo,Japan,"15,000",https://en.wikipedia.org/wiki/Dominate_World_Tour


In [0]:
# Salvar na camada silver 

df_clean.write.mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("skztour.02_silver.wikipedia_dominate")

print("Tabelas salvas com sucesso")
print(f"Total shows salvos: {df_clean.count()}")

Tabelas salvas com sucesso
Total shows salvos: 57
